## Train/Test Split

In this notebook, we merge the weather data and the bird observation data. We will split the data into a training set and a test set. Instead of choosing a random subset for the training set, we will perform the split based on the date of the observations.

### Importing the necessary modules

In [4]:
import pandas as pd
import numpy as np

import datetime

### Integrating the data

In [6]:
weather = pd.read_csv("Data/weather_two.txt")
weather["DATE"] = pd.to_datetime(weather[["YEAR", "MONTH", "DAY"]])
weather = weather.drop(["YEAR", "MONTH", "DAY"], axis = 1)

print(weather.info())
print(weather.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35768 entries, 0 to 35767
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   LATITUDE   35768 non-null  int64         
 1   LONGITUDE  35768 non-null  int64         
 2   PRCP       35768 non-null  float64       
 3   SNOW       35768 non-null  float64       
 4   TOBS       35768 non-null  float64       
 5   DATE       35768 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(2)
memory usage: 1.6 MB
None
   LATITUDE  LONGITUDE  PRCP  SNOW       TOBS       DATE
0        38        -90   0.0   0.0 -18.166667 2018-01-01
1        38        -89   0.0   0.0 -19.185714 2018-01-01
2        38        -88   0.0   0.0 -15.716667 2018-01-01
3        38        -87   0.0   0.0 -18.900000 2018-01-01
4        38        -86   0.0   0.0 -15.400000 2018-01-01


In [7]:
birds = pd.read_csv("Data/bird_two.txt")
birds["OBSERVATION DATE"] = pd.to_datetime(birds["OBSERVATION DATE"])

## convert the units for the time observations started to hour and round down to the nearest quarter of an hour 
birds["TIME OBSERVATIONS STARTED"] = (pd.to_timedelta(birds["TIME OBSERVATIONS STARTED"]) // np.timedelta64(15, 'm'))/4

print(birds.info())
print(birds.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 828198 entries, 0 to 828197
Data columns (total 14 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   OBSERVATION COUNT          828198 non-null  int64         
 1   IBA CODE                   828198 non-null  bool          
 2   BCR CODE                   828198 non-null  bool          
 3   USFWS CODE                 828198 non-null  bool          
 4   LATITUDE                   828198 non-null  float64       
 5   LONGITUDE                  828198 non-null  float64       
 6   OBSERVATION DATE           828198 non-null  datetime64[ns]
 7   TIME OBSERVATIONS STARTED  813921 non-null  float64       
 8   OBSERVATION TYPE           828198 non-null  object        
 9   DURATION MINUTES           814362 non-null  float64       
 10  EFFORT DISTANCE KM         465135 non-null  float64       
 11  EFFORT AREA HA             4474 non-null    float64 

                                   OBSERVATION COUNT  IBA CODE  BCR CODE  \
YEAR MONTH DAY LATITUDE LONGITUDE                                          
2017 1     1   28.0     -83.0                     41         0         4   
                        -82.0                    126         1        18   
                        -81.0                      3         1         1   
               29.0     -96.0                    258         2        23   
                        -95.0                      8         0         2   
...                                              ...       ...       ...   
2023 12    31  48.0     -90.0                    140         0         1   
                        -72.0                     30         0         1   
                        -71.0                     79         0         1   
                        -70.0                     28         2         3   
                        -69.0                     20         0         2   

           

Here we merge all the information by day

In [9]:
data = birds.merge(weather, 
                   how = "left", 
                   left_on = ["OBSERVATION DATE", "LATITUDE_ROUNDED", "LONGITUDE_ROUNDED"],
                   right_on = ["DATE", "LATITUDE", "LONGITUDE"],
                   suffixes = ("", "_x"))

data = data.drop(["LATITUDE_ROUNDED", "LONGITUDE_ROUNDED", "LATITUDE_x", "LONGITUDE_x", "DATE"], axis = 1)

# one hot encode the categorical data
data = pd.get_dummies(data, columns = ["OBSERVATION TYPE"])

# convert the entries in OBSERVATION DATE from datetime information to the number of days since January 1st, 2018 
def convertToNumDays(dt):
    return (np.datetime64(dt) - np.datetime64(datetime.datetime(2018,1,1))).astype('timedelta64[D]') // np.timedelta64(1, 'D')

data["OBSERVATION DATE"] = data["OBSERVATION DATE"].apply(convertToNumDays)

data = data.sort_values("OBSERVATION DATE", ignore_index = True)

print(data.info())
print(data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 828198 entries, 0 to 828197
Data columns (total 22 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   OBSERVATION COUNT                             828198 non-null  int64  
 1   IBA CODE                                      828198 non-null  bool   
 2   BCR CODE                                      828198 non-null  bool   
 3   USFWS CODE                                    828198 non-null  bool   
 4   LATITUDE                                      828198 non-null  float64
 5   LONGITUDE                                     828198 non-null  float64
 6   OBSERVATION DATE                              828198 non-null  int64  
 7   TIME OBSERVATIONS STARTED                     813921 non-null  float64
 8   DURATION MINUTES                              814362 non-null  float64
 9   EFFORT DISTANCE KM                            46

We can see that a significant portion of the data has missing values in the columns EFFORT DISTANCE KM and EFFORT AREA HA. We will simply drop the columns in order to deal with these missing values. It will be interesting to search for ways to impute the data in the future.

In [11]:
data = data.drop(["EFFORT DISTANCE KM", "EFFORT AREA HA"], axis = 1)
data = data.dropna(ignore_index = True)
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 811126 entries, 0 to 811125
Data columns (total 20 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   OBSERVATION COUNT                             811126 non-null  int64  
 1   IBA CODE                                      811126 non-null  bool   
 2   BCR CODE                                      811126 non-null  bool   
 3   USFWS CODE                                    811126 non-null  bool   
 4   LATITUDE                                      811126 non-null  float64
 5   LONGITUDE                                     811126 non-null  float64
 6   OBSERVATION DATE                              811126 non-null  int64  
 7   TIME OBSERVATIONS STARTED                     811126 non-null  float64
 8   DURATION MINUTES                              811126 non-null  float64
 9   PRCP                                          81

### Splitting the data into a training set and a test set

In [13]:
testCutoff = (np.datetime64(datetime.datetime(2019,11,1)) - np.datetime64(datetime.datetime(2018,1,1))).astype('timedelta64[D]') // np.timedelta64(1, 'D')

trainData = data[data["OBSERVATION DATE"] < testCutoff]
testData = data[data["OBSERVATION DATE"] >= testCutoff]

print(trainData.info())
print(trainData.tail())

print(testData.info())
print(testData.head())

<class 'pandas.core.frame.DataFrame'>
Index: 758683 entries, 0 to 758682
Data columns (total 20 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   OBSERVATION COUNT                             758683 non-null  int64  
 1   IBA CODE                                      758683 non-null  bool   
 2   BCR CODE                                      758683 non-null  bool   
 3   USFWS CODE                                    758683 non-null  bool   
 4   LATITUDE                                      758683 non-null  float64
 5   LONGITUDE                                     758683 non-null  float64
 6   OBSERVATION DATE                              758683 non-null  int64  
 7   TIME OBSERVATIONS STARTED                     758683 non-null  float64
 8   DURATION MINUTES                              758683 non-null  float64
 9   PRCP                                          758683 

In [14]:
trainData.to_csv("Data/TrainData.csv", index = False)
testData.to_csv("Data/TestData.csv", index = False)